<a href="https://colab.research.google.com/github/KN-Vignesh/AI-Projects/blob/main/QLora_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes>=0.46.1 torchao

In [ ]:
import torch


from datasets import Dataset


from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)


from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)


In [ ]:
training_data = [
    {
        "text": "Question: What are the working hours?\nAnswer: Our working hours are 9 AM to 6 PM."
    },
    {
        "text": "Question: How many annual leave days do employees get?\nAnswer: Employees receive 20 annual leave days."
    },
    {
        "text": "Question: Can employees work from home?\nAnswer: Employees can work from home two days per week."
    },
    {
        "text": "Question: What is the office dress code?\nAnswer: Employees should wear business casual clothing."
    },
    {
        "text": "Question: How do I apply for leave?\nAnswer: Employees should submit a leave request through the HR portal."
    },
    {
        "text": "Question: What time does the office open?\nAnswer: The office opens at 9 AM."
    }
]


dataset = Dataset.from_list(training_data)


In [ ]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
model = get_peft_model(model, lora_config)


In [ ]:
model.print_trainable_parameters()


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128
    )


tokenized_dataset = dataset.map(tokenize_function)


tokenized_dataset = tokenized_dataset.remove_columns(["text"])


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./qlora_hr_model",
    per_device_train_batch_size=2,
    num_train_epochs=10,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=True
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)


In [ ]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.595986
2,3.740661
3,3.825675
4,3.551318
5,4.025633
6,3.760118
7,3.593330
8,3.783469
9,3.937108
10,3.592246


TrainOutput(global_step=30, training_loss=3.6479841391245524, metrics={'train_runtime': 3.2403, 'train_samples_per_second': 18.517, 'train_steps_per_second': 9.258, 'total_flos': 332874547200.0, 'train_loss': 3.6479841391245524, 'epoch': 10.0})

In [ ]:
prompt = "Question: What are the working hours?\nAnswer:"


inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )


print(tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
))


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Question: What are the working hours?
Answer: I I I


,,,

,.. I and and


 or or,







